# Ch01-01: 다변량 데이터 시각화와 차원 해석


## 학습 목표

- 평행좌표(Parallel Coordinates)와 앤드류스 곡선(Andrews Curves)의 수학적 원리를 이해한다
- 조건부 플롯(Conditional Plot)으로 다변량 관계를 시각적으로 분해한다
- 다차원 스케일링(MDS)의 최적화 원리와 스트레스 함수를 유도한다
- t-SNE, UMAP과 MDS의 이론적 차이를 비교 분석한다


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

## 이론적 배경


### 1. 평행좌표 (Parallel Coordinates)

$p$차원 관측치 $\mathbf{x} = (x_1, \ldots, x_p)$를 $p$개의 수직 평행축 위의 점으로 매핑하고 직선으로 연결한다.

각 축은 정규화: $\tilde{x}_j = \frac{x_j - x_j^{\min}}{x_j^{\max} - x_j^{\min}}$

**핵심 성질**: 인접 축 간 교차 패턴은 변수 쌍의 **부호 상관**을 반영한다.

### 2. 앤드류스 곡선 (Andrews Curves)

$p$차원 벡터를 푸리에 기저로 1차원 함수로 변환:

$$f_{\mathbf{x}}(t) = \frac{x_1}{\sqrt{2}} + x_2\sin(t) + x_3\cos(t) + x_4\sin(2t) + x_5\cos(2t) + \cdots$$

등거리(isometric) 성질: $\int_{-\pi}^{\pi}[f_{\mathbf{x}}(t) - f_{\mathbf{y}}(t)]^2 dt = \pi\|\mathbf{x}-\mathbf{y}\|^2$

PCA 축 순서로 재배열하면 분산이 큰 성분이 저주파에 배치되어 해석력이 향상된다.

### 3. 다차원 스케일링 (MDS)

비유사도 행렬 $\boldsymbol{\Delta}=[\delta_{ij}]$로부터 저차원 좌표 $\mathbf{Y}$를 찾는다.

**Classical MDS**: 이중 중심화 + 고유값 분해

$$B = -\frac{1}{2}H\Delta^{(2)}H, \quad H = I_n - \frac{1}{n}\mathbf{1}\mathbf{1}^T$$

$$\mathbf{Y} = V_k\Lambda_k^{1/2}$$

**Stress (Kruskal)**:

$$\text{Stress}_1 = \sqrt{\frac{\sum_{i<j}(d_{ij}-\hat{d}_{ij})^2}{\sum_{i<j}d_{ij}^2}}$$

### 4. 조건부 플롯 (Coplot)

변수 $Z$를 겹치는 구간으로 분할하여 각 구간에서 $X$ vs $Y$ 관계를 패널로 시각화.
Simpson's paradox 탐지에 유용하다.


## 구현 가이드

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.manifold import MDS
from sklearn.preprocessing import StandardScaler
from pandas.plotting import parallel_coordinates, andrews_curves

# 데이터 준비
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

# 평행좌표
fig, ax = plt.subplots(figsize=(12, 5))
parallel_coordinates(df, 'species', ax=ax, colormap='viridis', alpha=0.4)
ax.set_title('평행좌표: Iris 데이터셋')
plt.tight_layout(); plt.show()

# 앤드류스 곡선
fig, ax = plt.subplots(figsize=(12, 5))
andrews_curves(df, 'species', ax=ax, colormap='viridis', alpha=0.3)
ax.set_title('앤드류스 곡선: Iris 데이터셋')
plt.tight_layout(); plt.show()

# Classical MDS 직접 구현
X_scaled = StandardScaler().fit_transform(iris.data)
from scipy.spatial.distance import pdist, squareform
D = squareform(pdist(X_scaled, 'euclidean'))
n = D.shape[0]
H = np.eye(n) - np.ones((n,n))/n
B = -0.5 * H @ (D**2) @ H
eigvals, eigvecs = np.linalg.eigh(B)
idx = np.argsort(eigvals)[::-1]
eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
Y_mds = eigvecs[:, :2] * np.sqrt(np.maximum(eigvals[:2], 0))

fig, ax = plt.subplots(figsize=(8, 6))
for i, name in enumerate(iris.target_names):
    mask = iris.target == i
    ax.scatter(Y_mds[mask, 0], Y_mds[mask, 1], label=name, alpha=0.7)
ax.set_xlabel('MDS Dim 1'); ax.set_ylabel('MDS Dim 2')
ax.set_title('Classical MDS'); ax.legend()
plt.tight_layout(); plt.show()


---
## 연습 문제

### 문제 1 [★]

6차원 합성 데이터(3클러스터, 각 100개)를 생성하고 평행좌표와 앤드류스 곡선을 그려라. 변수 순서를 PCA 기여도 순으로 재배열했을 때 앤드류스 곡선의 분리도가 개선되는지 비교하라.

> **힌트**: np.random.multivariate_normal로 클러스터를 생성하고 PCA로 변수 중요도 순서를 결정하세요.

In [ ]:
np.random.seed(42)
# TODO: 6차원 3클러스터 데이터
# TODO: 평행좌표
# TODO: 원래 vs PCA 순서 앤드류스 곡선 비교


### 문제 2 [★★]

Classical MDS를 직접 구현하라 (sklearn 금지).
1) Swiss Roll에 적용
2) Stress 값과 스크리 플롯
3) 유클리드 거리 vs 측지선(Isomap) 비교

> **힌트**: 측지선 거리는 k-NN 그래프에서 Dijkstra로 근사.

In [ ]:
from sklearn.datasets import make_swiss_roll
X_sr, color = make_swiss_roll(1000, noise=0.5, random_state=42)

def classical_mds(D, n_components=2):
    # TODO
    pass


### 문제 3 [★★]

조건부 플롯(Coplot)을 구현하라. 변수 Z를 겹치는 구간으로 분할하고 각 구간에서 X vs Y 산점도를 격자형으로 그려라. 합성 데이터에서 Simpson's paradox 유사 현상이 관측되는지 확인하라.

> **힌트**: 구간 겹침(overlap) 0.3~0.5로 설정.

In [ ]:
def coplot(df, x_col, y_col, z_col, n_panels=6, overlap=0.3):
    # TODO
    pass


### 문제 4 [★★★]

비선형 차원 축소(t-SNE, MDS)의 거리 보존 성능을 정량 비교하라.
1) 10차원 5클러스터 합성 데이터
2) Trustworthiness: $T(k)=1-\frac{2}{nk(2n-3k-1)}\sum_{i}\sum_{j\in U_i^k}(r(i,j)-k)$
3) Continuity, Shepard diagram 상관계수
4) k에 따른 변화 그래프

In [ ]:
from sklearn.manifold import TSNE, MDS
from sklearn.metrics import trustworthiness
# TODO: 데이터 생성, 임베딩, 지표 계산


---
## 참고 자료

- Inselberg, A. (2009). Parallel Coordinates: Visual Multidimensional Geometry.
- Borg, I. & Groenen, P. (2005). Modern Multidimensional Scaling.
- van der Maaten, L. & Hinton, G. (2008). Visualizing Data using t-SNE. JMLR.
- Andrews, D.F. (1972). Plots of High-Dimensional Data. Biometrics.
